In [1]:
%pip install tensorboard

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3.11 install --upgrade pip


In [4]:
!ls ../filestore/summarizationdata/

prepared


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import nltk
from sentence_transformers import SentenceTransformer, util
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from torch.utils.data import Subset
from datasets import load_from_disk
nltk.download('punkt')
nltk.download('punkt_tab')

/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[nltk_data] Downloading package punkt to /home/jupyter/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jupyter/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)
model_name = "google/mt5-large"
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto",
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 560/560 [00:03<00:00, 184.30it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 2,359,296 || all params: 1,744,169,984 || trainable%: 0.1353


In [7]:
train_dataset = load_from_disk("../filestore/summarizationdata/prepared/train")
valid_dataset = load_from_disk("../filestore/summarizationdata/prepared/valid")

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors='pt'
)

In [10]:
eval_dataset = Subset(valid_dataset, range(2000))

In [ ]:
seq_2_seq_args = Seq2SeqTrainingArguments(
    output_dir='./results_model',
    do_train=True,
    do_eval=True,
    do_predict=False,
    predict_with_generate=False,
    
    eval_strategy='steps',
    eval_steps=1000,
    
    save_strategy='steps',
    save_steps=500,
    save_total_limit=6,   
    
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    
    learning_rate=9e-5,
    num_train_epochs=1,
    warmup_steps=500,
    
    logging_steps=50, 
    log_level="info",
    report_to="tensorboard", 
    
    fp16=False,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)



trainer = Seq2SeqTrainer(
    model=model,
    args=seq_2_seq_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

***** Running training *****
  Num examples = 60,002
  Num Epochs = 1
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 4
  Total optimization steps = 7,501
  Number of trainable parameters = 2,359,296
  1%|          | 50/7501 [01:36<3:47:11,  1.83s/it]

{'loss': '120.7', 'grad_norm': '5.328e+04', 'learning_rate': '8.82e-06', 'epoch': '0.006666'}


  1%|▏         | 100/7501 [03:06<3:39:58,  1.78s/it]

{'loss': '122.8', 'grad_norm': '6.835e+04', 'learning_rate': '1.782e-05', 'epoch': '0.01333'}


  2%|▏         | 150/7501 [04:37<3:40:48,  1.80s/it]

...

 33%|███▎      | 2450/7501 [1:19:57<2:31:47,  1.80s/it]

{'loss': '10.12', 'grad_norm': '3.434', 'learning_rate': '6.495e-05', 'epoch': '0.3267'}


 33%|███▎      | 2500/7501 [1:21:28<2:31:31,  1.82s/it]Saving model checkpoint to ./results_model/checkpoint-2500


{'loss': '10.06', 'grad_norm': '3.65', 'learning_rate': '6.43e-05', 'epoch': '0.3333'}


loading configuration file config.json from cache at /tmp/xdg_cache/huggingface/hub/models--google--mt5-large/snapshots/50b7223e98fcd124b0cabb1ec81bc6324c7df107/config.json
Model config MT5Config {
  "architectures": [
    "MT5ForConditionalGeneration"
  ],
  "bos_token_id": null,
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "mt5",
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": true,
  "tokenizer_class": "T5Tokenizer",
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 250112